# Process the AGAP airborne magnetic data

We try to recreate the BAS processing steps.

In [1]:
%load_ext autoreload
%autoreload 2

import os

import geopandas as gpd
import pandas as pd

import airbornegeo

os.environ["POLARTOOLKIT_EPSG"] = "3031"

/home/sungw937/airbornegeo/src/airbornegeo/levelling.py:21: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


## Load data

This is a subset of the BAS AGAP survey over Antarctica's Gamburtsev Subglacial Mountains. The file is download and subset in the notebook `AGAP_magnetic_survey`.

In [2]:
data_df_full = pd.read_csv("../data/AGAP_magnetic_survey.csv")
print(data_df_full.columns)
data_df_full["index"] = data_df_full.index
data_df_full.head()

Index(['Lon', 'Lat', 'Height_WGS1984', 'Date', 'Time', 'MagR', 'MagC',
       'RefField', 'TCorr', 'MagRTC', 'BCorr_AGAP_N', 'BCorr_AGAP_NS',
       'BCorr_AGAP_S', 'BCorr_AGAP_SP', 'BCorr_Applied', 'MagBRTC', 'ACorr',
       'SCorr', 'MagF', 'MagL', 'MagML', 'easting', 'northing', 'line_name',
       'line', 'unixtime'],
      dtype='str')


,Lon,Lat,Height_WGS1984,Date,Time,MagR,MagC,RefField,TCorr,MagRTC,...,SCorr,MagF,MagL,MagML,easting,northing,line_name,line,unixtime,index
0,75.635600,-84.104393,4109.4,2008-12-17,0 days 07:53:42,55143.17,55143.10,55133.06,NaN,10.04,...,-32.96,-32.96,-22.30,NaN,621072.177354,159052.962392,DA500_11.0,1,1.229500e+09,0
1,75.636138,-84.103897,4111.5,2008-12-17,0 days 07:53:43,55142.15,55142.20,55133.03,NaN,9.19,...,-33.81,-33.81,-23.06,NaN,621126.010622,159060.533993,DA500_11.0,1,1.229500e+09,1
2,75.636699,-84.103403,4113.6,2008-12-17,0 days 07:53:44,55141.38,55141.30,55133.01,NaN,8.32,...,-34.68,-34.68,-23.81,NaN,621179.696916,159067.801202,DA500_11.0,1,1.229500e+09,2
3,75.637282,-84.102910,4115.4,2008-12-17,0 days 07:53:45,55140.60,55140.39,55132.98,NaN,7.44,...,-35.56,-35.56,-24.57,NaN,621233.338990,159074.801823,DA500_11.0,1,1.229500e+09,3
4,75.637876,-84.102417,4117.1,2008-12-17,0 days 07:53:46,55139.74,55139.48,55132.96,NaN,6.56,...,-36.44,-36.44,-25.32,NaN,621287.011834,159081.682095,DA500_11.0,1,1.229500e+09,4


In [3]:
# turn to geopandas geodataframe
data_df_full = gpd.GeoDataFrame(
    data_df_full,
    geometry=gpd.points_from_xy(x=data_df_full.easting, y=data_df_full.northing),
    crs="EPSG:3031",
)

In [4]:
# get only the raw columns
# we will perform the corrections ourselves and compare to their values
data_df = data_df_full.drop(columns=[])

# reset names to match the expected names in airbornegeo
data_df = data_df.rename(
    columns={
        "Height_WGS1984": "height",
    }
)

data_df.head()

,Lon,Lat,height,Date,Time,MagR,MagC,RefField,TCorr,MagRTC,...,MagF,MagL,MagML,easting,northing,line_name,line,unixtime,index,geometry
0,75.635600,-84.104393,4109.4,2008-12-17,0 days 07:53:42,55143.17,55143.10,55133.06,NaN,10.04,...,-32.96,-22.30,NaN,621072.177354,159052.962392,DA500_11.0,1,1.229500e+09,0,POINT (621072.177 159052.962)
1,75.636138,-84.103897,4111.5,2008-12-17,0 days 07:53:43,55142.15,55142.20,55133.03,NaN,9.19,...,-33.81,-23.06,NaN,621126.010622,159060.533993,DA500_11.0,1,1.229500e+09,1,POINT (621126.011 159060.534)
2,75.636699,-84.103403,4113.6,2008-12-17,0 days 07:53:44,55141.38,55141.30,55133.01,NaN,8.32,...,-34.68,-23.81,NaN,621179.696916,159067.801202,DA500_11.0,1,1.229500e+09,2,POINT (621179.697 159067.801)
3,75.637282,-84.102910,4115.4,2008-12-17,0 days 07:53:45,55140.60,55140.39,55132.98,NaN,7.44,...,-35.56,-24.57,NaN,621233.338990,159074.801823,DA500_11.0,1,1.229500e+09,3,POINT (621233.339 159074.802)
4,75.637876,-84.102417,4117.1,2008-12-17,0 days 07:53:46,55139.74,55139.48,55132.96,NaN,6.56,...,-36.44,-25.32,NaN,621287.011834,159081.682095,DA500_11.0,1,1.229500e+09,4,POINT (621287.012 159081.682)


In [8]:
airbornegeo.plotly_points(
    data_df[::50],
    color_col="line",
    hover_cols=[
        "line",
        "line_name",
        "unixtime",
    ],
    robust=False,
    size=3,
    edge_width=None,
)